# Instrumental variables

Balke–Pearl-style bounds for $Z \to X \to Y$ with latent confounding
$X \leftrightarrow Y$, plus multi-instrument and higher-cardinality variants.


In [ ]:
from partialid import Assumptions, CausalDiagram, Query, Variables, bound, bound_path
from partialid.problems import binary_iv, two_instruments

p = binary_iv(seed=2)
print(p.name, "—", p.notes)
print("query:", p.query.describe())


## Classic binary IV


In [ ]:
r = bound(p.variables, p.data, p.query, Assumptions.iv("Z", "X", "Y"))
print(r.summary(p.query))
print(f"tight={r.tight}  method={r.method}")
assert r.success and r.tight


## Optional no-defiers (`monotone=True`)

If compliance monotonicity conflicts with $P$, the LP is infeasible.


In [ ]:
r_m = bound(p.variables, p.data, p.query, Assumptions.iv("Z", "X", "Y", monotone=True))
print(f"monotone success={r_m.success}")
if not r_m.success:
    print("infeasible under no-defiers for this SCM sample")


## Same assumptions from an explicit diagram


In [ ]:
g = CausalDiagram.from_edges(
    nodes=["Z", "X", "Y"],
    directed=[("Z", "X"), ("X", "Y")],
    bidirected=[("X", "Y")],
)
r_diag = bound(p.variables, p.data, p.query, Assumptions.from_diagram(g))
print(r_diag.summary(p.query))
print("widths match:", abs(r_diag.width - r.width) < 1e-6)


## Two binary instruments $Z_1,Z_2 \to X \to Y$


In [ ]:
p2 = two_instruments(seed=3)
r2 = bound(p2.variables, p2.data, p2.query, p2.assumptions)
print(r2.summary(p2.query))
print(f"method={r2.method}  tight={r2.tight}  |cells|={len(p2.data)}")


## IV + Tan MSM at a fixed budget

Anchored MSM on the treatment can further shrink IV bounds when the budget is
feasible for $P$. (Very small $\Lambda$ may be infeasible for some SCM samples.)


In [ ]:
print("plain IV:", r.summary(p.query).splitlines()[1])
for lam in (2.5, 3.0, 4.0, 5.0):
    rr = bound(
        p.variables, p.data, p.query,
        Assumptions.iv("Z", "X", "Y") & Assumptions.tan_msm("X", lam),
    )
    if rr.success:
        print(f"IV + MSM Λ={lam:g}: [{rr.lower:.4f}, {rr.upper:.4f}] w={rr.width:.4f}  method={rr.method}")
    else:
        print(f"IV + MSM Λ={lam:g}: infeasible for this P")
